# WTA Tennis Daily Update

From [WTA Tennis daily update | Kaggle](https://www.kaggle.com/code/dissfya/wta-tennis-daily-update)

In [ ]:
import sys
import subprocess
from pathlib import Path
import datetime as dt
from io import BytesIO

import numpy as np
import pandas as pd

try:
    import requests
    import xlrd  # noqa: F401
    import openpyxl  # noqa: F401
except Exception:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "requests", "xlrd", "openpyxl"]
    )
    import requests

In [2]:
OUTPUT_CSV = Path("/kaggle/working/wta.csv")
INPUT_ROOT = Path("/kaggle/input")
INPUT_FILE_NAME = "wta.csv"

START_YEAR = 2007
CURRENT_YEAR = dt.date.today().year
UPDATE_YEAR = CURRENT_YEAR

MODE = "upsert-year"

In [3]:
def find_existing_csv(input_root=INPUT_ROOT, file_name=INPUT_FILE_NAME):
    candidates = sorted(input_root.glob(f"**/{file_name}"))
    return candidates[0] if candidates else None


def source_urls(year):
    preferred_ext = "xls" if year <= 2012 else "xlsx"
    fallback_ext = "xlsx" if preferred_ext == "xls" else "xls"
    hosts = [
        "https://www.tennis-data.co.uk",
        "http://www.tennis-data.co.uk",
        "https://tennis-data.co.uk",
        "http://tennis-data.co.uk",
    ]
    urls = []
    for ext in [preferred_ext, fallback_ext]:
        urls.extend([f"{host}/{year}w/{year}.{ext}" for host in hosts])
    return urls


def read_existing_dataset(path):
    if path is None or not Path(path).exists():
        return pd.DataFrame()
    df = pd.read_csv(path, low_memory=False)
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df


def download_file(url, retries=4, timeout=90):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Kaggle WTA daily update)",
        "Accept": "application/vnd.ms-excel,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,*/*",
    }
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            if not response.content:
                raise ValueError("Empty response")
            return BytesIO(response.content)
        except Exception as exc:
            last_error = exc
            wait_seconds = min(2**attempt, 30)
            print(f"Download failed ({attempt}/{retries}): {url} -> {exc}")
            if attempt < retries:
                import time

                time.sleep(wait_seconds)

    raise RuntimeError(f"Could not download {url}: {last_error}")


def read_year(year):
    errors = []
    for url in source_urls(year):
        try:
            buffer = download_file(url)
            df = pd.read_excel(buffer)
            df["SourceYear"] = year
            print(f"Loaded {year}: {url}")
            return df
        except Exception as exc:
            errors.append(f"{url}: {exc}")

    message = "\n".join(errors[-4:])
    raise RuntimeError(f"Failed to load WTA year {year}. Last errors:\n{message}")


def read_years(years):
    frames = []
    for year in years:
        try:
            frames.append(read_year(year))
        except Exception as exc:
            print(f"Skipped {year}: {exc}")
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [4]:
def ensure_columns(df, columns, default=np.nan):
    for col in columns:
        if col not in df.columns:
            df[col] = default
    return df


def clean_numeric(series, default=np.nan):
    return pd.to_numeric(
        series.replace(r"^\s*$", np.nan, regex=True), errors="coerce"
    ).fillna(default)


def fill_b365_odds(df):
    win_cols = [
        c
        for c in ["CBW", "EXW", "PSW", "UBW", "LBW", "SJW", "MaxW", "AvgW"]
        if c in df.columns
    ]
    lose_cols = [
        c
        for c in ["CBL", "EXL", "PSL", "UBL", "LBL", "SJL", "MaxL", "AvgL"]
        if c in df.columns
    ]

    if "B365W" not in df.columns:
        df["B365W"] = np.nan
    if "B365L" not in df.columns:
        df["B365L"] = np.nan

    if win_cols:
        df["B365W"] = df["B365W"].fillna(
            df[win_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        )
    if "AvgW" in df.columns:
        df["B365W"] = df["B365W"].fillna(pd.to_numeric(df["AvgW"], errors="coerce"))

    if lose_cols:
        df["B365L"] = df["B365L"].fillna(
            df[lose_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        )
    if "AvgL" in df.columns:
        df["B365L"] = df["B365L"].fillna(pd.to_numeric(df["AvgL"], errors="coerce"))

    return df


def build_score(df):
    p1_winner = df["ind"].eq(0)

    parts = []
    for set_no in [1, 2, 3]:
        w = df[f"W{set_no}"].astype(int).astype(str)
        l = df[f"L{set_no}"].astype(int).astype(str)
        p1_score = np.where(p1_winner, w + "-" + l, l + "-" + w)
        parts.append(pd.Series(p1_score, index=df.index))

    score = parts[0] + " " + parts[1] + " " + parts[2]
    return (
        score.str.replace(r"\b0-0\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def transform(raw_df):
    if raw_df.empty:
        return pd.DataFrame()

    df = raw_df.copy()
    required = [
        "Tournament",
        "Date",
        "Court",
        "Surface",
        "Round",
        "Best of",
        "Winner",
        "Loser",
        "WRank",
        "LRank",
        "WPts",
        "LPts",
        "W1",
        "L1",
        "W2",
        "L2",
        "W3",
        "L3",
        "B365W",
        "B365L",
        "Comment",
    ]
    df = ensure_columns(df, required)

    df["Best of"] = df["Best of"].fillna(3)
    df = df[df["Comment"].eq("Completed")].copy()

    for col in ["WRank", "LRank", "W1", "W2", "L1", "L2"]:
        df = df[df[col].notna()].copy()

    if df.empty:
        return pd.DataFrame()

    df = fill_b365_odds(df)
    df[["W3", "L3"]] = df[["W3", "L3"]].fillna(0)

    score_cols = ["W1", "L1", "W2", "L2", "W3", "L3"]
    for col in score_cols:
        df[col] = clean_numeric(df[col], default=0).astype(int)

    df = df.reset_index(drop=True)
    df["ind"] = np.arange(len(df)) % 2
    p1_winner = df["ind"].eq(0)

    df["Player_1"] = np.where(p1_winner, df["Winner"], df["Loser"])
    df["Player_2"] = np.where(p1_winner, df["Loser"], df["Winner"])
    df["Rank_1"] = np.where(p1_winner, df["WRank"], df["LRank"])
    df["Rank_2"] = np.where(p1_winner, df["LRank"], df["WRank"])
    df["Pts_1"] = np.where(p1_winner, df["WPts"], df["LPts"])
    df["Pts_2"] = np.where(p1_winner, df["LPts"], df["WPts"])
    df["Odd_1"] = np.where(p1_winner, df["B365W"], df["B365L"])
    df["Odd_2"] = np.where(p1_winner, df["B365L"], df["B365W"])
    df["Score"] = build_score(df)

    output_cols = [
        "Tournament",
        "Date",
        "Court",
        "Surface",
        "Round",
        "Best of",
        "Player_1",
        "Player_2",
        "Winner",
        "Rank_1",
        "Rank_2",
        "Pts_1",
        "Pts_2",
        "Odd_1",
        "Odd_2",
        "Score",
    ]
    out = df[output_cols].copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.fillna(-1)

    int_cols = ["Best of", "Rank_1", "Rank_2", "Pts_1", "Pts_2"]
    for col in int_cols:
        out[col] = clean_numeric(out[col].replace("NR", -1), default=-1).astype(int)

    out = out.sort_values(
        ["Date", "Tournament", "Round", "Player_1", "Player_2"], na_position="last"
    )
    return out.reset_index(drop=True)

In [5]:
def update_dataset(output_path=OUTPUT_CSV, year=UPDATE_YEAR, mode=MODE):
    existing_path = find_existing_csv()
    existing = read_existing_dataset(existing_path)
    existing_rows = len(existing)
    updated_year_rows = 0
    old_current_year_rows = 0
    status = "updated"

    if existing.empty:
        print("Existing dataset not found. Building full history.")
        raw = read_years(range(START_YEAR, year + 1))
        if raw.empty:
            raise RuntimeError("No WTA source files could be loaded.")
        final = transform(raw)
        updated_year_rows = (
            len(final[final["Date"].dt.year.eq(year)]) if "Date" in final.columns else 0
        )
    else:
        print(f"Existing dataset: {existing_path}")
        if "Date" in existing.columns:
            existing["Date"] = pd.to_datetime(existing["Date"], errors="coerce")
            old_current_year_rows = int(existing["Date"].dt.year.eq(year).sum())

        try:
            raw = read_year(year)
            updated_year = transform(raw)
            updated_year_rows = len(updated_year)
        except Exception as exc:
            print(f"Current year refresh failed: {exc}")
            print("Keeping existing dataset unchanged for this run.")
            final = existing.copy()
            status = "unchanged"
        else:
            if mode == "upsert-year":
                old = existing[existing["Date"].dt.year.ne(year)].copy()
                final = pd.concat([old, updated_year], ignore_index=True)
            elif mode == "append-new":
                key = ["Date", "Tournament", "Round", "Player_1", "Player_2", "Winner"]
                old_keys = (
                    existing[key].astype(str).agg("||".join, axis=1)
                    if set(key).issubset(existing.columns)
                    else pd.Series([], dtype=str)
                )
                new_keys = updated_year[key].astype(str).agg("||".join, axis=1)
                final = pd.concat(
                    [existing, updated_year[~new_keys.isin(set(old_keys))]],
                    ignore_index=True,
                )
            else:
                raise ValueError("mode must be 'upsert-year' or 'append-new'")

    final["Date"] = pd.to_datetime(final["Date"], errors="coerce")
    before_dedup_rows = len(final)
    final = final.drop_duplicates(
        subset=["Date", "Tournament", "Round", "Player_1", "Player_2", "Winner"],
        keep="last",
    )
    dropped_duplicates = before_dedup_rows - len(final)
    final = final.sort_values(
        ["Date", "Tournament", "Round", "Player_1", "Player_2"], na_position="last"
    )
    final.to_csv(output_path, index=False)

    final_rows = len(final)
    delta = final_rows - existing_rows
    final_current_year_rows = (
        int(final["Date"].dt.year.eq(year).sum()) if "Date" in final.columns else 0
    )

    print("Update summary")
    print(f"Status: {status}")
    print(f"Mode: {mode}")
    print(f"Year: {year}")
    print(f"Rows before: {existing_rows:,}")
    print(f"Rows in old current year: {old_current_year_rows:,}")
    print(f"Rows loaded for current year: {updated_year_rows:,}")
    print(f"Rows in final current year: {final_current_year_rows:,}")
    print(f"Dropped duplicates: {dropped_duplicates:,}")
    print(f"Rows after: {final_rows:,}")
    print(f"Delta: {delta:+,}")
    print(f"Output file: {output_path}")

    return final.reset_index(drop=True)


updated = update_dataset()
updated.tail()

Existing dataset: /kaggle/input/wta-tennis-2007-2023-daily-update/wta.csv
Download failed (1/4): https://www.tennis-data.co.uk/2026w/2026.xlsx -> HTTPSConnectionPool(host='www.tennis-data.co.uk', port=443): Max retries exceeded with url: /2026w/2026.xlsx (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:997)')))
Download failed (2/4): https://www.tennis-data.co.uk/2026w/2026.xlsx -> HTTPSConnectionPool(host='www.tennis-data.co.uk', port=443): Max retries exceeded with url: /2026w/2026.xlsx (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:997)')))
Download failed (3/4): https://www.tennis-data.co.uk/2026w/2026.xlsx -> HTTPSConnectionPool(host='www.tennis-data.co.uk', port=443): Max retries exceeded with url: /2026w/2026.xlsx (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:997)')))
Download failed (4/4): https://www.tennis-data.co.

,Tournament,Date,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
45378,Monterrey Open,2026-08-29,Outdoor,Hard,Semifinals,3,Bartunkova N.,Mertens E.,Mertens E.,38,24,1237,1870,2.3,1.62,6-1 4-6 2-6
45379,Monterrey Open,2026-08-29,Outdoor,Hard,Semifinals,3,Parry D.,Li A.,Parry D.,50,30,1113,1518,2.38,1.57,7-6 6-4
45380,Monterrey Open,2026-08-30,Outdoor,Hard,The Final,3,Parry D.,Mertens E.,Parry D.,50,24,1113,1870,2.1,1.73,6-4 0-6 6-3
45381,Landsky Lighting Guangzhou International Women...,NaT,Outdoor,Hard,The Final,3,Groth J.,Kudryavtseva A.,Groth J.,55,103,1105,666,1.16,4.5,6-1 6-4
45382,Western & Southern Financial Group Women's Open,NaT,Outdoor,Hard,The Final,3,Kerber A.,Li N.,Li N.,7,9,5225,3795,2.0,1.8,6-1 3-6 1-6


In [6]:
check = pd.read_csv(OUTPUT_CSV, low_memory=False)
print(check.shape)
check.tail()

(45383, 16)


,Tournament,Date,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
45378,Monterrey Open,2026-08-29,Outdoor,Hard,Semifinals,3,Bartunkova N.,Mertens E.,Mertens E.,38,24,1237,1870,2.3,1.62,6-1 4-6 2-6
45379,Monterrey Open,2026-08-29,Outdoor,Hard,Semifinals,3,Parry D.,Li A.,Parry D.,50,30,1113,1518,2.38,1.57,7-6 6-4
45380,Monterrey Open,2026-08-30,Outdoor,Hard,The Final,3,Parry D.,Mertens E.,Parry D.,50,24,1113,1870,2.1,1.73,6-4 0-6 6-3
45381,Landsky Lighting Guangzhou International Women...,NaN,Outdoor,Hard,The Final,3,Groth J.,Kudryavtseva A.,Groth J.,55,103,1105,666,1.16,4.5,6-1 6-4
45382,Western & Southern Financial Group Women's Open,NaN,Outdoor,Hard,The Final,3,Kerber A.,Li N.,Li N.,7,9,5225,3795,2.0,1.8,6-1 3-6 1-6
